<h1>AI-Based Resume Screening System</h1>

In [1]:
!pip install PyPDF2 
!pip install python-docx


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import PyPDF2
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import docx
import os


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from PyPDF2 import PdfReader
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
def extract_text_from_pdf(pdf_path):
    text = ""
    reader = PdfReader(pdf_path)
    for page in reader.pages:
        text += page.extract_text()
    return text   

In [6]:
def extract_text_from_docx(docx_path):
    document = docx.Document(docx_path)
    text = ""

    for paragraph in document.paragraphs:
        text += paragraph.text + " "

    return text

In [7]:
def extract_resume_text(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".pdf":
        return extract_text_from_pdf(file_path)

    elif extension == ".docx":
        return extract_text_from_docx(file_path)

    else:
        raise ValueError("Only PDF and DOCX files are supported.")

In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s+#.]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [9]:
sample_text = "Python Developer! Skilled in SQL, Machine Learning & Power BI."
print(clean_text(sample_text))

python developer skilled in sql machine learning power bi.


In [10]:
job_description = """
Job Title: Data Analyst

Company: DataSphere Analytics Pvt. Ltd.

Job Description:
 DataSphere Analytics is looking for a Data Analyst to collect, clean, analyze, and visualize data to support business decisions.

Responsibilities:
- Collect, clean, and analyze large datasets.
- Create dashboards and reports using Power BI or Tableau.
- Perform SQL queries to extract data from databases.
- Use Python (Pandas, NumPy, Matplotlib) for data analysis.
- Identify trends, patterns, and business insights.
- Collaborate with business teams to understand requirements.
- Prepare reports and present findings.

Required Skills:
- Python
- SQL
- Excel
- Power BI
- Tableau
- Pandas
- NumPy
- Data Analysis
- Data Visualization
- Statistics
- Machine Learning (Basic)

Qualification:
Bachelor's degree in Computer Science, Information Technology, Data Science, Statistics, or a related field.

Experience:
0–2 years (Freshers can apply).
"""

cleaned_job_description = clean_text(job_description)
print(cleaned_job_description)

job title data analyst company datasphere analytics pvt. ltd. job description datasphere analytics is looking for a data analyst to collect clean analyze and visualize data to support business decisions. responsibilities collect clean and analyze large datasets. create dashboards and reports using power bi or tableau. perform sql queries to extract data from databases. use python pandas numpy matplotlib for data analysis. identify trends patterns and business insights. collaborate with business teams to understand requirements. prepare reports and present findings. required skills python sql excel power bi tableau pandas numpy data analysis data visualization statistics machine learning basic qualification bachelor s degree in computer science information technology data science statistics or a related field. experience 0 2 years freshers can apply .


In [11]:
resume_path = "NikitaBhosale (2).pdf"

resume_text = extract_resume_text(resume_path)
cleaned_resume = clean_text(resume_text)

print(cleaned_resume[:500])

nikita sham bhosale satara maharashtra +91 9404835103 bhosalenikita258 gmail.com professional summary entry level data analyst with hands on experience in python sql and power bi through projects professional training and internship. skilled in data cleaning exploratory data analysis eda and data visualization. strong analytical mindset with a chemistry background and a passion for converting data into actionable insights. technical skills programming data analysis python pandas numpy eda sql jo


In [12]:
documents = [
    cleaned_resume,
    cleaned_job_description
]

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(documents)

print("TF-IDF matrix created successfully")
print("Matrix shape:", tfidf_matrix.shape)

TF-IDF matrix created successfully
Matrix shape: (2, 256)


In [13]:
similarity_score = cosine_similarity(
    tfidf_matrix[0:1],
    tfidf_matrix[1:2]
)

match_percentage = similarity_score[0][0] * 100

print(f"Resume Match Score: {match_percentage:.2f}%")

Resume Match Score: 43.89%


In [14]:
required_skills = [
    "python", "sql", "excel", "power bi", "tableau",
    "pandas", "numpy", "data analysis",
    "data visualization", "statistics",
    "machine learning", "communication", "problem solving"
]

matched_skills = []
missing_skills = []

for skill in required_skills:
    if skill in cleaned_resume:
        matched_skills.append(skill)
    else:
        missing_skills.append(skill)

print("Matched Skills:")
print(matched_skills)

print("\nMissing Skills:")
print(missing_skills)

Matched Skills:
['python', 'sql', 'excel', 'power bi', 'tableau', 'pandas', 'numpy', 'data analysis', 'data visualization', 'statistics', 'machine learning']

Missing Skills:
['communication', 'problem solving']


In [15]:
skill_match_percentage = (
    len(matched_skills) / len(required_skills)
) * 100

final_score = (
    0.70 * skill_match_percentage +
    0.30 * match_percentage
)

print(f"TF-IDF Score: {match_percentage:.2f}%")
print(f"Skill Match Score: {skill_match_percentage:.2f}%")
print(f"Final Weighted Score: {final_score:.2f}%")

TF-IDF Score: 43.89%
Skill Match Score: 84.62%
Final Weighted Score: 72.40%


In [16]:
##final Screening
if final_score >= 75:
    status = "Selected for Interview"
elif final_score >= 60:
    status = "Shortlisted"
elif final_score >= 40:
    status = "Consider for Review"
else:
    status = "Rejected"

print("=" * 40)
print("AI RESUME SCREENING RESULT")
print("=" * 40)
print(f"Resume Match Score      : {match_percentage:.2f}%")
print(f"Skill Match Score       : {skill_match_percentage:.2f}%")
print(f"Final Weighted Score    : {final_score:.2f}%")
print(f"Matched Skills          : {len(matched_skills)}")
print(f"Missing Skills          : {len(missing_skills)}")
print(f"Status                  : {status}")

print("\nMatched Skills:")
print(", ".join(matched_skills))

print("\nMissing Skills:")
print(", ".join(missing_skills))

AI RESUME SCREENING RESULT
Resume Match Score      : 43.89%
Skill Match Score       : 84.62%
Final Weighted Score    : 72.40%
Matched Skills          : 11
Missing Skills          : 2
Status                  : Shortlisted

Matched Skills:
python, sql, excel, power bi, tableau, pandas, numpy, data analysis, data visualization, statistics, machine learning

Missing Skills:
communication, problem solving
